# Nyaya Mitra — GRPO training on Colab Pro

Runbook for the 60-credit ($60) HF budget. Uses Colab A100 large (~$4/hr equivalent). Phase 1 + Phase 2 + eval rerun + plot regeneration.

**Prerequisites:** Colab Pro with A100 selected (`Runtime > Change runtime type > A100`). HF token with read access. Optional W&B key.

**Total wall-clock:** ~12-15h. Budget pause-and-resume between phases.

## 1. Verify GPU + clone repo

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/parthtaneja0001/nyaya-mitra.git
%cd /content/nyaya-mitra

## 2. Install deps

Unsloth wheel pinned to a known-good combo with Colab's CUDA. ~3 min.

In [ ]:
!pip install -q --no-deps 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q -e '.[track_a,track_b,dev]'
!pip install -q vllm wandb

## 3. Auth

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HF token: ")
wb = getpass("W&B API key (optional, blank to skip): ").strip()
if wb:
    os.environ["WANDB_API_KEY"] = wb
!huggingface-cli login --token $HF_TOKEN

## 4. Sanity: 192 track-b tests still green

In [ ]:
!pytest tests/track_b -q --tb=no

## 5. Smoke run (~1h, ~$4 of budget)

Qwen 2.5 0.5B + 50 episodes. Verifies the full pipeline works before burning real credits on phase 1.

In [ ]:
!bash scripts/run_smoke.sh

Inspect the smoke metrics. Reward should be a finite float; gates may fire (model is tiny). If no traceback, you're good.

In [ ]:
import json

rows = [
    json.loads(line)
    for line in open("training/dumps/smoke_metrics.jsonl").read().splitlines()
    if line.strip()
]
print(f"rows: {len(rows)}")
print(rows[-1])

## 6. Phase 1 warmup (~6h, ~$24)

Qwen 2.5 3B Instruct, 500 episodes. Target rolling-100 mean ≥ 0.30.

In [ ]:
!bash scripts/run_phase1.sh

Inspect phase 1 results. If rolling-100 mean is below 0.30, **stop** — don't run phase 2. Instead jump to Section 8 to eval the phase-1 adapter and figure out what's broken.

In [ ]:
import json

rows = [
    json.loads(line)
    for line in open("training/dumps/phase1_metrics.jsonl").read().splitlines()
    if line.strip()
]
print(f"phase 1 episodes: {len(rows)}")
print(f"final rolling mean (100): {rows[-1].get('rolling_mean_100', 0):.3f}")
print(f"last 5 rewards: {[r['total_reward'] for r in rows[-5:]]}")

## 7. Phase 2 co-training (~5h, ~$20)

Skip if phase 1 didn't reach 0.30 rolling mean. Otherwise:

In [ ]:
!bash scripts/run_phase2.sh

## 8. Eval + plot regeneration (~30min, ~$2)

Use whichever final adapter you have.

In [ ]:
import os

phase2 = "training/checkpoints/adapter_final_phase2.lora"
phase1 = "training/checkpoints/adapter_final_warmup.lora"
adapter = phase2 if os.path.isdir(phase2) else phase1
label = "phase2" if adapter == phase2 else "phase1"
print(f"evaluating {label} adapter at {adapter}")
!bash scripts/run_eval_post_train.sh {adapter} {label}

## 9. Render the headline plots inline

In [ ]:
from IPython.display import Image, display

for name in [
    "total_reward_curve.png",
    "baseline_vs_trained_bars.png",
    "integration_solve_rate.png",
]:
    print(f"=== {name} ===")
    display(Image(f"demo/plots/{name}"))

## 10. Commit + push artifacts

Push the trained adapter, training metrics jsonl, regenerated plots, and updated reports back to the repo so judges can clone-and-replay.

In [ ]:
!git config user.email 'parth.sankhla98@gmail.com'
!git config user.name 'Parth Sankhla'
!git add training/dumps/ demo/plots/ eval/report_*.md
!git status
# uncomment when ready:
# !git commit -m 'real training run: phase 1 + phase 2 artifacts'
# !git push origin HEAD

## 11. Deploy env to HF Space (~30min)

In [ ]:
import os

os.environ["HF_SPACE_REPO"] = "parthtaneja0001/nyaya-mitra-env"
!bash scripts/deploy_space.sh